# ASCII Maze RL — H100 GRPO Demo

GRPO training on 8×8 and 9×9 mazes using the strong SFT base model.
This notebook runs on an H100 GPU and is projected at the front of the room.

The SFT model solves 3×3–7×7 perfectly but fails on 8×8/9×9.
GRPO's job: push 8×8 from 0% to positive through exploration and reward shaping.

In [ ]:
# Install deps — avoid overwriting Colab's CUDA PyTorch
!pip install -q trl peft datasets accelerate bitsandbytes 2>&1 | tail -1

# Verify CUDA torch is still intact
import torch
if not torch.cuda.is_available():
    print("⚠️  CUDA not available! Restart runtime: Runtime → Restart session")
    print("   Then re-run all cells from the top.")
else:
    print(f"✓ CUDA OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the maze infrastructure
!git clone -q -b claude-trl https://github.com/StephenJHardy/ascii-maze-rl.git
import sys
sys.path.insert(0, 'ascii-maze-rl')

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")

## 1. Explore the Maze Format

Let's see what mazes look like and how solutions work.

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str, to_prompt, SYSTEM_PROMPT, solution_to_str
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress
from src.reward import compute_reward

# Generate a sample maze
maze = generate(4, 4, seed=42)
print("Maze (4×4):")
print(to_str(maze))
print(f"\nSolution: {solution_to_str(maze.solution_moves)}")
print(f"Solution length: {len(maze.solution_moves)} moves")
print(f"\nEntry: {maze.entry}, Exit: {maze.exit}")

In [ ]:
# See how the prompt looks (this is what the model receives)
print("System prompt:")
print(SYSTEM_PROMPT)
print("\nUser message (just the maze grid):")
print(to_str(maze))

In [ ]:
# See how move parsing and simulation work
test_output = "r r d d d r"
moves = extract_moves(test_output)
print(f"Parsed moves: {moves}")

path = simulate(moves, maze)
print(f"Path through maze: {path}")
print(f"Reached exit: {reached_exit(path, maze)}")
print(f"Valid steps: {len(path) - 1} / {len(moves)}")
print(f"Progress toward exit: {manhattan_progress(path[-1], maze.exit, maze.entry):.2f}")

## 2. Generate Training & Evaluation Data

In [ ]:
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig, MazeRecord

# GRPO training data: 7×7 (partial success) + 8×8/9×9 (target)
train_config = DatasetConfig(
    name="grpo_train",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=7, height=7, count=500, start_seed=840_000),
        SizeConfig(width=8, height=8, count=1000, start_seed=850_000),
        SizeConfig(width=9, height=9, count=1000, start_seed=860_000),
    ],
)
train_ds = MazeDataset.generate(train_config, progress=False)
print(f"Training: {train_ds.summary()}")

# Evaluation data (disjoint seeds, all sizes for regression check)
eval_config = DatasetConfig(
    name="eval",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=5, height=5, count=20, start_seed=900_000),
        SizeConfig(width=6, height=6, count=20, start_seed=910_000),
        SizeConfig(width=7, height=7, count=30, start_seed=920_000),
        SizeConfig(width=8, height=8, count=40, start_seed=930_000),
        SizeConfig(width=9, height=9, count=40, start_seed=940_000),
    ],
)
eval_ds = MazeDataset.generate(eval_config, progress=False)
print(f"Eval: {eval_ds.summary()}")

In [ ]:
from datasets import Dataset

# Rollout viewer config
ROLLOUT_SIZES = [
    SizeConfig(width=7, height=7, count=5, start_seed=970_000),
    SizeConfig(width=8, height=8, count=10, start_seed=980_000),
    SizeConfig(width=9, height=9, count=5, start_seed=990_000),
]
MAX_ROLLOUT_TOKENS = 100

def maze_records_to_hf_dataset(records):
    rows = []
    for r in records:
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": r.maze_str},
        ]
        rows.append({
            "prompt": prompt,
            "maze_id": r.id,
            "solution_moves": r.solution_moves,
            "solution_length": r.solution_length,
            "walls": r.walls,
            "entry": r.entry,
            "exit": r.exit,
            "width": r.width,
            "height": r.height,
            "solution_path": r.solution_path,
        })
    return Dataset.from_list(rows)

# Convert training data to HF format for TRL
train_hf = maze_records_to_hf_dataset(train_ds.records)
print(f"HF training dataset: {len(train_hf)} mazes")

## 3. Load the Pre-trained Model

We load `Qwen2.5-0.5B-Instruct` with a pre-trained SFT LoRA adapter
that already knows the maze format and can solve ~68% of 4×4 mazes.

Your GRPO reward function will push this higher.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# SFT model (LoRA merged into base weights)
model_id = "StephenJHardy/maze-cuda-sft-9x9-qwen2.5-0.5b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Loaded SFT model from {model_id}")

## 4. Test the Model Before GRPO

Let's see how the model performs before RL training.

In [ ]:
def generate_solution(model, tokenizer, maze, max_new_tokens=100):
    """Generate a solution for a maze using the model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": to_str(maze)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True,
    )
    return response


def evaluate_model(model, tokenizer, eval_dataset, max_new_tokens=100):
    """Quick evaluation — returns solve rate by size."""
    results = {}
    for record in eval_dataset:
        maze = record.to_maze()
        response = generate_solution(model, tokenizer, maze, max_new_tokens)
        moves = extract_moves(response)
        path = simulate(moves or [], maze)
        solved = reached_exit(path, maze)
        size = f"{record.width}x{record.height}"
        if size not in results:
            results[size] = {"total": 0, "solved": 0}
        results[size]["total"] += 1
        results[size]["solved"] += int(solved)
    return results


# Quick test on a few mazes
for seed in [42, 99, 7]:
    maze = generate(4, 4, seed=seed)
    response = generate_solution(model, tokenizer, maze)
    reward = compute_reward(response, maze)
    print(f"Maze 4x4_{seed}: output={response!r:40s} reward={reward:.3f}")

In [ ]:
# Full evaluation before GRPO
print("Evaluating model before GRPO...")
pre_results = evaluate_model(model, tokenizer, eval_ds.records)
print("\nPre-GRPO results:")
for size, stats in sorted(pre_results.items()):
    rate = stats['solved'] / stats['total'] * 100
    print(f"  {size}: {stats['solved']}/{stats['total']} ({rate:.0f}%)")

## 5. Reward Function

The reward function scores each rollout. It uses partial credit:
negative for gibberish, proportional to valid moves and progress
for partial solutions, and a bonus for reaching the exit.

In [ ]:
from src.maze_gen import Maze


def reconstruct_maze(walls, entry, exit_, width, height, solution_path):
    """Reconstruct a Maze object from dataset metadata."""
    frozen_walls = frozenset(
        frozenset(tuple(c) for c in w) for w in walls
    )
    return Maze(
        width=width,
        height=height,
        walls=frozen_walls,
        entry=tuple(entry),
        exit=tuple(exit_),
        solution=tuple(tuple(p) for p in solution_path),
        seed=0,
    )


def maze_reward(completions, **kwargs):
    """
    Reward function for GRPO maze training.
    Partial credit design with negative penalty for gibberish.
    """
    rewards = []

    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            completion = completion[-1]["content"] if completion else ""
        elif isinstance(completion, dict):
            completion = completion.get("content", "")

        maze = reconstruct_maze(
            walls=kwargs["walls"][i],
            entry=kwargs["entry"][i],
            exit_=kwargs["exit"][i],
            width=kwargs["width"][i],
            height=kwargs["height"][i],
            solution_path=kwargs["solution_path"][i],
        )

        moves = extract_moves(completion)

        if moves is None:
            rewards.append(-1.0)
            continue

        path = simulate(moves, maze)
        valid_steps = len(path) - 1
        optimal_len = len(maze.solution) - 1

        if reached_exit(path, maze):
            efficiency = optimal_len / max(len(moves), optimal_len)
            rewards.append(0.6 + 0.4 * efficiency)
            continue

        coverage = min(valid_steps / max(optimal_len, 1), 1.0)
        progress = manhattan_progress(path[-1], maze.exit, maze.entry)
        progress = max(progress, 0.0)
        partial = 0.5 * (0.7 * coverage + 0.3 * progress)
        rewards.append(partial)

    return rewards

In [ ]:
# Test your reward function on some examples
maze = generate(4, 4, seed=42)
correct = solution_to_str(maze.solution_moves)

test_completions = [
    correct,                    # perfect solution
    "r r d d",                  # wrong moves (may hit wall)
    correct + " u u u",         # correct + extra junk
    "I cannot solve this maze", # gibberish
    "",                         # empty
]

# Build kwargs as TRL would pass them
n = len(test_completions)
test_kwargs = {
    "walls": [MazeRecord.from_maze(maze).walls] * n,
    "entry": [list(maze.entry)] * n,
    "exit": [list(maze.exit)] * n,
    "width": [maze.width] * n,
    "height": [maze.height] * n,
    "solution_path": [[list(p) for p in maze.solution]] * n,
}

rewards = maze_reward(test_completions, **test_kwargs)
print("Reward function test:")
for comp, rew in zip(test_completions, rewards):
    print(f"  {rew:+.3f}  {comp!r:.50s}")

print("\n💡 If all non-solved outputs score the same, GRPO won't learn much.")
print("   Try adding partial credit so 'almost right' scores higher than 'gibberish'.")

## 6. Train with GRPO

Now we run GRPO using your reward function. The trainer will:
1. Sample prompts from the training set
2. Generate multiple completions per prompt
3. Score them with your reward function
4. Update the model toward higher-reward completions

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

grpo_config = GRPOConfig(
    output_dir="./maze-grpo-h100",
    num_train_epochs=1,
    max_steps=200,
    per_device_train_batch_size=16,
    num_generations=16,
    max_completion_length=100,  # 8×8/9×9 need longer solutions
    learning_rate=5e-6,
    beta=0.04,
    logging_steps=10,
    save_steps=100,
    temperature=1.0,
    report_to="none",
    bf16=True,
    gradient_accumulation_steps=1,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[maze_reward],
    args=grpo_config,
    train_dataset=train_hf,
    peft_config=peft_config,
)

print("Starting GRPO training on 8×8/9×9 mazes...")
print(f"  Steps: {grpo_config.max_steps}")
print(f"  Generations per prompt: {grpo_config.num_generations}")
print(f"  Max completion tokens: {grpo_config.max_completion_length}")

In [ ]:
trainer.train()

## Explore Rollouts

Capture GRPO-style rollouts and visualize them. This shows what
GRPO sees at each step: multiple attempts at the same maze,
their rewards, and which ones get reinforced.

In [ ]:
from src.rollout_capture import capture_rollouts_pytorch, save_rollouts
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig

# Generate a small eval set for rollout visualization
rollout_config = DatasetConfig(
    name="rollout_viz",
    algorithm="wilson",
    sizes=ROLLOUT_SIZES,
)
rollout_ds = MazeDataset.generate(rollout_config, progress=False)

# Capture rollouts from the trained model
print("Capturing rollouts from trained model...")
trained_model = trainer.model
trained_model.eval()
rollouts = capture_rollouts_pytorch(
    trained_model, tokenizer, rollout_ds.records,
    num_generations=8, temperature=1.0, max_new_tokens=MAX_ROLLOUT_TOKENS,
)
save_rollouts(rollouts, 'rollouts.json')
print(f"Captured {sum(len(r.rollouts) for r in rollouts)} rollouts across {len(rollouts)} mazes")

In [ ]:
# Build and display the rollout viewer
from src.build_rollout_viewer import HTML_TEMPLATE
import json as _json
from dataclasses import asdict
from IPython.display import HTML, display

viewer_data = [asdict(r) for r in rollouts]
serialized = _json.dumps(viewer_data).replace('</','<\\/')
viewer_html = HTML_TEMPLATE.replace('__DATA_PLACEHOLDER__', serialized)

# Save to file (downloadable) and display inline
with open('rollout_viewer.html', 'w') as f:
    f.write(viewer_html)
print('Viewer saved to rollout_viewer.html')

# Display inline (limited height)
display(HTML(f'<iframe srcdoc="{viewer_html.replace(chr(34), "&quot;")}" width="100%" height="800px" style="border:none;"></iframe>'))

## 7. Evaluate After GRPO

In [ ]:
# Evaluate the trained model
trained_model = trainer.model

print("Evaluating model after GRPO...")
post_results = evaluate_model(trained_model, tokenizer, eval_ds.records)

print("\nResults comparison:")
print(f"{'Size':>5s}  {'Before':>8s}  {'After':>8s}  {'Change':>8s}")
print("-" * 35)
for size in sorted(set(list(pre_results.keys()) + list(post_results.keys()))):
    pre = pre_results.get(size, {"solved": 0, "total": 0})
    post = post_results.get(size, {"solved": 0, "total": 0})
    pre_rate = pre["solved"] / max(pre["total"], 1) * 100
    post_rate = post["solved"] / max(post["total"], 1) * 100
    diff = post_rate - pre_rate
    print(f"{size:>5s}  {pre_rate:7.1f}%  {post_rate:7.1f}%  {diff:+7.1f}pp")

In [ ]:
# What is the model actually outputting now?
print("Sample outputs after GRPO:\n")
for record in eval_ds.records[:10]:
    maze = record.to_maze()
    response = generate_solution(trained_model, tokenizer, maze)
    print(f"  {record.id}: {response!r:.60s}")

print("\n" + "="*60)
print("If results got WORSE, your reward function likely has a problem.")
print("Common issues with the naive binary reward (1.0/0.0):")
print("  - All 8 generations score 0.0 → advantages are all zero")
print("  - GRPO gets no gradient signal → model drifts randomly")
print("  - Over many steps, drift destroys the SFT baseline")
print("")
print("Fix: Go back to Section 5 and add PARTIAL CREDIT:")
print("  - Negative reward for gibberish (not zero!)")
print("  - Partial credit proportional to valid moves")
print("  - Progress toward exit as a smooth signal")
print("="*60)

In [ ]:
# Look at some specific examples
print("\nSample outputs after GRPO:\n")
for seed in [42, 99, 7, 123, 456]:
    for size in [4, 5]:
        maze = generate(size, size, seed=seed)
        response = generate_solution(trained_model, tokenizer, maze)
        moves = extract_moves(response)
        path = simulate(moves or [], maze)
        solved = reached_exit(path, maze)
        status = "✓" if solved else "✗"
        correct = solution_to_str(maze.solution_moves)
        print(f"  {status} {size}x{size}_{seed}: model={response!r:30s} correct={correct}")